In [ ]:
!nvidia-smi | head -12

# GPU benchmark: TokenVector.Inference vs PyTorch trên Colab GPU

So sánh train GPT-2 124M (random init, không cần fixture) forward+backward trên:
- **.tkv** — `demo/bseq.dll` (IL portable) + `tkv_simd_native` (C, AVX2) + dispatch cuBLAS nếu `libcublas.so.12` load được
- **PyTorch** — CUDA (chỉ để tham khảo)

Cùng seq1024, mode 0 = fwd+bwd (không optimizer) để apple-to-apple.
Cell `[5/5]` tách 5 pha (fwd / CE-loss / bwd / opt / copyback, `.tkv` mode 2)
và chạy PyTorch `tier_e` với **2 thread** (đúng số thread GEMM pool của `.tkv`)
để biết pha nào đáng tối ưu.

> **Trạng thái 2026-11-12 (đo trên T4):** `TkvCUDA_Available() = 1` (cuBLAS nạp được)
> nhưng GPU **thực tế không được dùng** — hook cuBLAS chỉ nằm trong kernel tile
> `SgemmF32(a,b,o,T)` với `T < 512`, và mỗi lần gọi lại `cudaMalloc` + copy.
> Kết quả `.tkv` dưới đây là **CPU** (2 vCPU của Colab), không phải GPU.
>
> Trên Linux, kernel ASM AVX2 (Windows x64 ABI) không dùng được — engine tự rơi về
> bản vô hướng cho các op phần tử, còn GEMM/attention vẫn chạy trong C `.so`.
> Phép so sánh đúng là **.tkv CPU** vs **PyTorch CPU** (cùng VM); số PyTorch
> CUDA chỉ để tham khảo.

In [ ]:
# [1/5] Install .NET 10 SDK (full SDK, includes runtime) + required libs
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
!for u in https://dot.net/v1/dotnet-install.sh https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh; do curl -fsSL --retry 5 --retry-delay 3 --connect-timeout 20 "$u" -o /tmp/dotnet-install.sh && break; echo "that bai: $u"; done; ls -l /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --info | head -15


In [ ]:
# [2/5] Clone repos + CUDA runtime libs (cublas/cudart) + LD_LIBRARY_PATH
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git
!pip install -q nvidia-cublas-cu12 nvidia-cuda-runtime-cu12 2>&1 | tail -2
import os, glob
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
cudart_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
print('cublas_dir:', cublas_dir)
print('cudart_dir:', cudart_dir)
if cublas_dir:
    os.environ['LD_LIBRARY_PATH'] = cublas_dir + ':' + (cudart_dir or '') + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# [3/5] Build native .so (C + AVX2 + CUDA dispatch) và kiểm tra CUDA khả dụng
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC \
  tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c \
  -lm -lpthread -ldl -o tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/demo/tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/tkv_simd_native.dll
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll
!cp TokenVector/Release/tools/simd_native/tkv_step124.dll TokenVector.Inference/demo/tkv_step124.dll
print('Built .so')
import ctypes
lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
lib.TkvCUDA_Available.restype = ctypes.c_int
print('TkvCUDA_Available() =', lib.TkvCUDA_Available(), '(1 = cuBLAS loaded, 0 = CPU fallback)')


In [ ]:
# [4/5] Bench .tkv vs PyTorch (seq1024, fwd+bwd) — CPU và CUDA
# Ghi chu 2026-11-13: cell nay chay MAC DINH tren Linux = BLAS ON (cap 8).
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt
# sampler có timeout (KHÔNG dùng pkill -f nvidia-smi: nó giết luôn shell của cell)
!nohup timeout 400 nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader --loop=2 > /content/gpu.log 2>&1 & echo sampler_started
!cd TokenVector.Inference && $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | tail -3
!echo '--- GPU trong luc .tkv chay (moi dong = 1 mau 2s) ---'; sort -u /content/gpu.log | head -6
print('--- PyTorch CPU (cung VM, apples-to-apples voi .tkv CPU) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu python3 benchmark/bench_torch.py d 1024 2>&1 | tail -3
print('--- PyTorch CUDA (T4) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024 2>&1 | tail -3
!echo '--- GPU trong luc torch CUDA chay ---'; tail -12 /content/gpu.log
!nvidia-smi --query-gpu=memory.used --format=csv,noheader


In [ ]:
# [5/5] Phase breakdown: fwd / CE-loss / bwd / opt / copyback (mode 2)
!cd TokenVector.Inference && printf '1024\n2\n1\n' > tv/tests/_seq_cfg.txt
print('--- .tkv native (TKV_GEMM_BLAS=0, kernel AVX2 noi bo) ---')
!cd TokenVector.Inference && TKV_GEMM_BLAS=0 $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE' 
print('--- .tkv MAC DINH (BLAS ON + TKV_GEMM_BLAS_THREADS=8, tu 2026-11-13) ---')
!cd TokenVector.Inference && $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE|BLAS'
!nohup timeout 300 nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader --loop=2 > /content/gpu_cublas.log 2>&1 & echo cublas_sampler_started
print('--- .tkv + cuBLAS (TKV_GEMM_CUDA=1; buffer device duoc cache) ---')
!cd TokenVector.Inference && TKV_GEMM_CUDA=1 $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE|BLAS|CUDA'
import os; print('--- GPU trong luc .tkv cuBLAS chay ---'); print('\n'.join(sorted(set(open('/content/gpu_cublas.log').read().splitlines()))))
print('--- PyTorch CPU 2 thread, tach pha tuong ung ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 python3 benchmark/bench_torch.py e 1024 2>&1 | tail -6


In [ ]:
# [7/7] Peak RAM (VmHWM) cua .tkv vs PyTorch CPU - cung seq1024, mode 0 (fwd+bwd, khong optimizer)
!apt-get install -y -qq time >/dev/null 2>&1 || true
print('--- .tkv MAC DINH (BLAS), mode 0 seq1024, VmPeak/VmHWM ---')
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt && /usr/bin/time -v $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BSEQ|Maximum resident'
print('--- PyTorch CPU (tier_d = fwd+bwd (T=2), seq1024) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 /usr/bin/time -v python3 benchmark/bench_torch.py d 1024 2>&1 | grep -E 'TOTAL|Maximum resident'


In [ ]:
# [8/8] Apples-to-apples: .tkv vs PyTorch - xen ke A/B/A/B, 3 vong moi ben, median
# Dieu kien: cung VM, seq1024, GPT-2 124M, input token [(i*37+11)%50257] giong nhau,
# seed 7 ca hai phia, 2 thread (TKV_BENCH_THREADS=2 = pool GEMM cua .tkv), n=3 moi ben.
# Luu y fairness: params khoi tao KHAC nhau (loss khong so duoc); chi so TIME.
import subprocess, re, statistics as st
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run_tkv(cuda):
    env = 'TKV_GEMM_CUDA=1' if cuda else ''
    cmd = 'cd TokenVector.Inference && printf "1024\n0\n1\n" > tv/tests/_seq_cfg.txt && ' + env + ' ' + DOT + ' demo/bseq.dll'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=900).stdout
    m = re.search(r'fwd=([\d.]+) ms  bwd=([\d.]+) ms  total=([\d.]+)', out)
    return tuple(float(x) for x in m.groups())
def run_torch(dev):
    cmd = 'cd TokenVector.Inference && TKV_BENCH_DEVICE=' + dev + ' TKV_BENCH_THREADS=2 python3 benchmark/bench_torch.py d 1024'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=900).stdout
    m = re.search(r'fwd\+bwd seq1024 dev=' + dev + r': median=([\d.]+)', out)
    return float(m.group(1))
def pair(name, cuda):
    tk=[]; tt=[]
    for i in range(3):
        a = run_tkv(cuda); b = run_torch('cuda' if cuda else 'cpu')
        tk.append(a[2]); tt.append(b)
        print('%s vong %d: .tkv total=%.1f ms (fwd=%.0f bwd=%.0f)  torch=%.1f ms' % (name, i+1, a[2], a[0], a[1], b))
    print('%s: .tkv median=%.1f ms | torch median=%.1f ms | ty le=%.2fx' % (name, st.median(tk), st.median(tt), st.median(tk)/st.median(tt)))
pair('CPU', False)
pair('CUDA', True)
print('Xong. Dieu kien: nhu in o dau cell; moi ben do 3 lan xen ke de tich VM drift.')
